# Synthetic AI-detection benchmark workflow

This notebook uses synthetic multilingual samples to demonstrate a multi-detector benchmark workflow. It provides pluggable detector backends, an offline mock detector, output normalization into a common table, and basic cross-detector agreement summaries.

It contains no API keys, private corpora or paths, detector-accuracy claims, or publication-sensitive outputs from the original research workflow.

## Data boundary

All examples use synthetic data generated in this notebook. If a real detector backend is added, provide credentials through environment variables or a local secrets manager; do not hard-code them.

In [ ]:
import os
import re
import hashlib
from dataclasses import dataclass
from typing import Dict, List

import numpy as np
import pandas as pd

## Synthetic benchmark set

Labels identify demonstration scenarios only; they are not empirical ground truth for any real detector.

In [ ]:
SYNTHETIC_SAMPLES = [
    {
        "sample_id": "ar_human_01",
        "language": "Arabic",
        "label": "Human-written (synthetic scenario)",
        "text": "في هذا المثال التركيبي نصف خطوات تجربة صغيرة. بدأ الباحث بتنظيف النص، ثم راجع النتائج يدوياً، ثم كتب ملاحظات قصيرة عن حدود العينة. لا يحتوي هذا النص على بيانات حقيقية أو أسماء أشخاص أو ملفات خاصة.",
    },
    {
        "sample_id": "ar_ai_01",
        "language": "Arabic",
        "label": "AI-generated (synthetic scenario)",
        "text": "يقدم هذا النص عرضاً منظماً للغاية مع انتقالات متشابهة وإيقاع ثابت وجمل متقاربة الطول. يكرر الفكرة نفسها بصياغات متعددة ويعطي انطباعاً عاماً بالاتساق الشديد دون تفاصيل سياقية كثيرة.",
    },
    {
        "sample_id": "en_human_01",
        "language": "English",
        "label": "Human-written (synthetic scenario)",
        "text": "I drafted these notes after comparing two preprocessing options. The first preserved punctuation but kept noisy spacing. The second normalized whitespace and made later parsing easier, so I kept it and documented the tradeoff.",
    },
    {
        "sample_id": "en_ai_01",
        "language": "English",
        "label": "AI-generated (synthetic scenario)",
        "text": "This passage is intentionally polished, evenly structured, and broadly explanatory. It summarizes a process in a smooth and generic way, with repeated framing phrases and limited concrete detail about any specific run or dataset.",
    },
    {
        "sample_id": "code_human_01",
        "language": "Code",
        "label": "Human-written code (synthetic scenario)",
        "text": "def rolling_mean(values, window):\n    if window <= 0:\n        raise ValueError('window must be positive')\n    out = []\n    for i in range(len(values)):\n        start = max(0, i - window + 1)\n        chunk = values[start:i+1]\n        out.append(sum(chunk) / len(chunk))\n    return out\n",
    },
    {
        "sample_id": "code_ai_01",
        "language": "Code",
        "label": "AI-generated code (synthetic scenario)",
        "text": 'def process_data(items):\n    results = []\n    for item in items:\n        transformed_item = str(item).strip().lower()\n        if transformed_item:\n            results.append({"value": transformed_item, "status": "processed"})\n    return results\n',
    },
]

samples_df = pd.DataFrame(SYNTHETIC_SAMPLES)
samples_df[["sample_id", "language", "label"]]

## Shared helper functions

In [ ]:
def count_words(text: str) -> int:
    if not text:
        return 0
    return len(re.findall(r"\S+", text, flags=re.UNICODE))


def normalize_percent(value) -> float:
    try:
        value = float(value)
    except (TypeError, ValueError):
        return np.nan

    if value <= 1.0:
        value *= 100.0
    return round(value, 2)


def label_from_percent(percent: float, threshold: float = 50.0) -> str:
    if pd.isna(percent):
        return "Unknown"
    return "Likely AI" if percent >= threshold else "Likely Human"


def stable_unit_interval(text: str, salt: str) -> float:
    digest = hashlib.sha256((salt + "||" + text).encode("utf-8")).hexdigest()
    return int(digest[:12], 16) / float(16**12 - 1)

## Detector interface

The original notebook queried external services and normalized their outputs to a common schema. This version uses a small interface with an offline mock backend.

In [ ]:
@dataclass
class DetectorResult:
    detector_name: str
    ai_rate_percent: float
    result_label: str
    raw: Dict[str, object]


class BaseDetector:
    name = "base"

    def predict(self, text: str, language: str) -> DetectorResult:
        raise NotImplementedError


class MockDetector(BaseDetector):
    def __init__(self, name: str, salt: str, bias: float = 0.0):
        self.name = name
        self.salt = salt
        self.bias = bias

    def predict(self, text: str, language: str) -> DetectorResult:
        base = stable_unit_interval(text + "|" + language, self.salt)

        punctuation_density = min(
            1.0, len(re.findall(r"[,:;()]", text)) / max(1, len(text) / 40)
        )
        repeated_tokens = len(re.findall(r"\b(\w+)\b(?=.*\b\1\b)", text.lower()))
        repetition_signal = min(1.0, repeated_tokens / 20.0)
        code_signal = 1.0 if re.search(r"def |return |for |if |\{|\}|=>", text) else 0.0

        score = (
            0.45 * base
            + 0.20 * repetition_signal
            + 0.10 * punctuation_density
            + 0.10 * code_signal
            + self.bias
        )
        score = max(0.0, min(1.0, score))
        percent = normalize_percent(score)

        return DetectorResult(
            detector_name=self.name,
            ai_rate_percent=percent,
            result_label=label_from_percent(percent),
            raw={
                "mock_score_unit_interval": round(score, 4),
                "language": language,
            },
        )


detectors = [
    MockDetector(name="gptzero_mock", salt="gptzero", bias=0.05),
    MockDetector(name="pangram_mock", salt="pangram", bias=-0.02),
    MockDetector(name="sapling_mock", salt="sapling", bias=0.00),
    MockDetector(name="isgen_mock", salt="isgen", bias=0.03),
]

## Credential configuration

This cell demonstrates configuration access only; it does not call an external service.

In [ ]:
REAL_BACKEND_ENV_VARS = {
    "gptzero": os.getenv("GPTZERO_API_KEY"),
    "pangram": os.getenv("PANGRAM_API_KEY"),
    "sapling": os.getenv("SAPLING_API_KEY"),
    "isgen": os.getenv("ISGEN_API_KEY"),
}

pd.Series(
    {k: v is not None for k, v in REAL_BACKEND_ENV_VARS.items()}, name="configured"
)

## Synthetic benchmark

In [ ]:
rows: List[Dict[str, object]] = []

for sample in SYNTHETIC_SAMPLES:
    row = {
        "sample_id": sample["sample_id"],
        "language": sample["language"],
        "label": sample["label"],
        "word_count": count_words(sample["text"]),
    }

    for detector in detectors:
        result = detector.predict(sample["text"], sample["language"])
        row[f"{detector.name}_ai_rate_percent"] = result.ai_rate_percent
        row[f"{detector.name}_result"] = result.result_label

    rows.append(row)

results_df = pd.DataFrame(rows)
results_df

## Agreement summary

This summary applies only to mock outputs and does not characterize real detector behavior.

In [ ]:
result_cols = [c for c in results_df.columns if c.endswith("_result")]
rate_cols = [c for c in results_df.columns if c.endswith("_ai_rate_percent")]

summary_df = results_df[["sample_id", "language", "label"]].copy()
summary_df["mean_ai_rate_percent"] = results_df[rate_cols].mean(axis=1).round(2)
summary_df["detectors_likely_ai"] = (results_df[result_cols] == "Likely AI").sum(axis=1)
summary_df["detectors_likely_human"] = (results_df[result_cols] == "Likely Human").sum(
    axis=1
)
summary_df = summary_df.sort_values(["language", "sample_id"]).reset_index(drop=True)
summary_df

Use a long-format (tidy) table for aggregation, visualization, or export.

In [ ]:
long_rows = []

for sample in SYNTHETIC_SAMPLES:
    for detector in detectors:
        result = detector.predict(sample["text"], sample["language"])
        long_rows.append(
            {
                "sample_id": sample["sample_id"],
                "language": sample["language"],
                "scenario_label": sample["label"],
                "detector": detector.name,
                "ai_rate_percent": result.ai_rate_percent,
                "result": result.result_label,
            }
        )

long_df = pd.DataFrame(long_rows)
long_df.head(12)

## Production backend notes

If publication is permitted, move detector implementations to `src/pipeline/` and limit notebooks to orchestration and visualization. Log request failures, rate limits, and schema-normalized outputs; do not commit raw private text.